# 01. Checkout de e-commerce: planejar, medir e reduzir variância

**Setor:** e-commerce de eletrônicos. **Decisão:** vale a pena lançar o novo
checkout de 1 clique? Queremos o efeito na **receita por usuário**.

Este caso percorre um fluxo completo de teste A/B online:

1. **Planejar** o tamanho da amostra (`power_analysis`) a partir da variância
   histórica da receita.
2. **Desenhar** com randomização completa (`CRD`) e **checar SRM** antes de
   olhar qualquer resultado.
3. **Estimar** o efeito com a variância de Neyman (população finita) e comparar
   com o bootstrap (superpopulação).
4. **Reduzir variância** com `CUPED`, usando o gasto pré-experimento como
   covariável.

Base teórica: [II. Fundamentos e desenhos](../../../docs/pt-br/teoria/01-fundamentos.md),
[IV. Inferência](../../../docs/pt-br/teoria/04-inferencia.md) (tópicos 14, 15 e 17)
e [III. Estimação](../../../docs/pt-br/teoria/03-estimacao.md) (tópico 11, CUPED).


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


def _find_data():
    """Locate examples/use_cases/data whether run from the notebook or the root."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / "data", base / "examples" / "use_cases" / "data"):
            if (cand / "ecommerce_checkout.csv").exists():
                return cand
    raise FileNotFoundError("Could not locate examples/use_cases/data")


DATA = _find_data()

df = pd.read_csv(DATA / "ecommerce_checkout.csv")
print(df.shape)
df.head()


## 1. Quantos usuários? (poder)

A receita por usuário tem uma cauda longa (poucos usuários gastam muito), então
sua variância é alta. Usamos o desvio padrão histórico (a coluna `revenue_y0`, o
comportamento sem tratamento) para dimensionar o teste para um efeito mínimo
detectável (MDE) de `+2,5` na receita, com poder 80% e `alpha = 0,05`.


In [ ]:
from skxperiments.design.power import power_analysis

std_hist = float(df["revenue_y0"].std())
plan = power_analysis(mde=2.5, power=0.8, std=std_hist, alpha=0.05)
print(f"historical std: {std_hist:.1f}")
print(f"required n_total: {plan.n_total}  (~{plan.n_treated} per arm)")
print(f"we collected {len(df)} users, comfortably above the plan")


In [ ]:
from skxperiments.reporting import plot_power_curve

n_values = list(range(200, 4001, 200))
ax = plot_power_curve(n_values, mde=2.5, std=std_hist, alpha=0.05)
ax.set_title("Power vs. sample size (MDE = 2.5)")
ax.figure


## 2. Desenho e checagem de SRM

Randomizamos os usuários 50/50 com `CRD`. Depois construímos o resultado
observado: cada usuário revela `revenue_y1` se caiu no tratamento e `revenue_y0`
caso contrário (a "tabela da ciência" do dado sintético). Antes de estimar
qualquer coisa, rodamos o `SRMTest`: se a proporção observada não bate com a
planejada, algo quebrou e nada mais é confiável.


In [ ]:
from skxperiments.core.assignment import CRDAssignment
from skxperiments.design.crd import CRD
from skxperiments.diagnostics import SRMTest

design = CRD(p=0.5, seed=101)
assignment = design.randomize(df[["device", "sessions_pre", "spend_pre"]].copy())

t = assignment.data_[assignment.treatment_col_].to_numpy()
data = assignment.data_.copy()
data["revenue"] = np.where(t == 1, df["revenue_y1"].to_numpy(), df["revenue_y0"].to_numpy())
assignment = CRDAssignment(
    data=data, treatment_col=assignment.treatment_col_, design=design, seed=101
)

srm = SRMTest().run(assignment)
print(f"SRM flagged: {srm.flagged}  (p={srm.p_value:.3f})")


## 3. Estimar o efeito e reduzir a variância

- `DifferenceInMeans` + `NeymanCI`: o ponto e o intervalo de confiança na visão
  de **população finita** (SATE).
- `BootstrapCI`: a visão de **superpopulação** (PATE), reamostrando dentro de
  cada braço.
- `CUPED`: usa o gasto pré-experimento (`spend_pre`), correlacionado com a
  receita, para remover ruído. O erro padrão cai sem introduzir viés.


In [ ]:
from skxperiments.estimators.difference_in_means import DifferenceInMeans
from skxperiments.estimators.cuped import CUPED
from skxperiments.inference import NeymanCI, BootstrapCI

neyman = NeymanCI(estimator=DifferenceInMeans(outcome_col="revenue")).fit(assignment).estimate()
boot = BootstrapCI(
    estimator=DifferenceInMeans(outcome_col="revenue"), n_resamples=1000, seed=0
).fit(assignment).estimate()
cuped = BootstrapCI(
    estimator=CUPED(outcome_col="revenue", pre_experiment_col="spend_pre"),
    n_resamples=1000, seed=0,
).fit(assignment).estimate()

pd.DataFrame([
    {"method": "DIM / Neyman (SATE)", "ATE": neyman.ate, "SE": neyman.se,
     "CI_low": neyman.ci[0], "CI_high": neyman.ci[1]},
    {"method": "DIM / Bootstrap (PATE)", "ATE": boot.ate, "SE": boot.se,
     "CI_low": boot.ci[0], "CI_high": boot.ci[1]},
    {"method": "CUPED / Bootstrap", "ATE": cuped.ate, "SE": cuped.se,
     "CI_low": cuped.ci[0], "CI_high": cuped.ci[1]},
]).round(3)


In [ ]:
from skxperiments.reporting import plot_effect

ax = plot_effect(neyman)
ax.set_title("Revenue lift (Neyman 95% CI)")
ax.figure


## Decisão

O efeito verdadeiro embutido no dado é `+2,5`. As três abordagens concordam no
ponto (perto de `+2,5`), e o intervalo exclui zero: o novo checkout **aumenta a
receita**. O `CUPED` entrega o **menor erro padrão** (o gasto pré-experimento
explica boa parte da variância da receita), então é a leitura mais precisa para
sustentar a decisão de lançar.

Próximo passo natural: [02. Lojas com blocagem](02_fashion_stores_blocking.ipynb).
